[CAPSTONE INTEGRATION TARGET]
    Carry measured device dimensions from JSON into Spreadsheet aliases and expressions. Prove that a changed measurement updates the model and that a recompute restores the requested final dimensions.


# 1. Dimensions are data
    Numeric input is checked before it enters an executable FreeCAD script. JSON keys are a fixed schema, and values are millimetres. Device names and notes belong in an evidence record, not executable source.


In [ ]:
from __future__ import annotations
import importlib.util
import json
import math
import os
from pathlib import Path
import sys

if sys.version_info < (3, 12):
    raise RuntimeError("Choose the CortexCAD Python 3.12+ Jupyter kernel")

# Installed packages work from any directory. A source checkout can be selected
# explicitly; ancestor discovery also supports launching Jupyter inside a phase.
if importlib.util.find_spec("cortexcad") is None:
    root_hint: str = os.environ.get("CORTEXCAD_ROOT", "")
    candidates: list[Path] = ([Path(root_hint).expanduser()] if root_hint else [])
    candidates += [Path.cwd(), *Path.cwd().parents]
    source_root: Path | None = next((path for path in candidates
                                    if (path / "cortexcad" / "recipes.py").is_file()), None)
    if source_root is None:
        raise RuntimeError("Install CortexCAD in this kernel or set CORTEXCAD_ROOT to its checkout")
    sys.path.insert(0, str(source_root.resolve()))

from cortexcad.client import FreeCADClient
from cortexcad.recipes import DEFAULTS, build_script, validate_parameters
LIVE: bool = os.environ.get("CORTEXCAD_LIVE") == "1"
print("LIVE execution enabled" if LIVE else "OFFLINE: no CAD connection will be opened")


In [ ]:
def read_numeric_object(text: str) -> dict[str, object]:
    """Reject ambiguous duplicate JSON keys before phase-specific validation."""
    def unique_pairs(pairs: list[tuple[str, object]]) -> dict[str, object]:
        result: dict[str, object] = {}
        for key, value in pairs:
            if key in result:
                raise ValueError(f"Duplicate JSON key: {key}")
            result[key] = value
        return result
    def reject_constant(value: str) -> object:
        raise ValueError(f"Non-finite JSON constant: {value}")
    value: object = json.loads(text, object_pairs_hook=unique_pairs, parse_constant=reject_constant)
    if not isinstance(value, dict):
        raise ValueError("Measurements must be a JSON object")
    return value


In [ ]:
PHASE: str = "parametric"
json_payload: str = '{"length": 60, "width": 40, "height": 6}'
params: dict[str, float] = validate_parameters(PHASE, read_numeric_object(json_payload))
expected_volume: float = math.prod(params[key] for key in ("length", "width", "height"))
print("Requested final volume:", expected_volume, "mm³")
print("Expected temporary 90% length volume:", expected_volume * 0.9, "mm³")


## 2. Inspect the dependency graph
    B1/B2/B3 are aliased MountLength/MountWidth/MountHeight. A Part Box refers to `Dimensions.MountLength`, `Dimensions.MountWidth`, and `Dimensions.MountHeight`. The native script checks each dimension, temporarily reduces length to 90%, recomputes, checks its volume, restores the requested value, recomputes again and commits.

    The intermediate update is a native integration check. Merely accepting JSON or returning a successful TCP connection cannot establish a working expression graph.


In [ ]:
script: str = build_script(PHASE, params)
compile(script, "freecad_lesson.py", "exec")
print(script)
print("Compiled source only: no FreeCAD solver or solid has run on this host.")


In [ ]:
if LIVE:
    # Endpoint and explicit timeout come from CORTEXCAD_ENDPOINT/TIMEOUT.
    client: FreeCADClient = FreeCADClient(timeout=float(os.environ.get("CORTEXCAD_TIMEOUT", "5")))
    outcome = client.run_recipe(PHASE, params)
    print(repr(outcome))
    print("Inspect the workstation and native assertions before accepting this lesson.")
else:
    print("OFFLINE preview complete; native geometry and hardware-fit gates remain pending.")


## 3. Reject ambiguous or unsafe payloads
    Duplicate JSON keys silently overwrite earlier values in many parsers. This notebook rejects them. Nonfinite constants, booleans, strings and unknown keys also fail before any request.


In [ ]:
for bad_json in ('{"length": 60, "length": 80}', '{"length": NaN}', '{"length": "__import__(\"os\")"}', '{"length": true}', '{"unexpected": 1}'):
    try:
        validate_parameters(PHASE, read_numeric_object(bad_json))
    except ValueError as error:
        print("Expected rejection:", error)
    else:
        raise AssertionError("Unsafe payload accepted")


## 4. Exercise and evidence
    Set dimensions to 80 × 50 × 8 mm. Predict a final 32,000 mm³ and intermediate 28,800 mm³. Inspect `ExpressionEngine`, the real dimensions and native volume after an accepted live run. In a disposable copy, remove Height's expression and show that a B3 update no longer drives the solid.

    Keep unit, instrument, uncertainty, reviewer and hardware identity in a separate measurement record. Proceed to the capstone only when measured geometry is available; the supplied fixture is an example.
